In [1]:
from bs4 import BeautifulSoup
import requests
import time
import pandas as pd
import numpy as np
import yfinance as yf
import html5lib

from playwright.sync_api import sync_playwright
from playwright.async_api import async_playwright
import asyncio
import nest_asyncio

# **I- SCRAPPING DES DATAS DES DIFFIRENTES STOCKS DES INDICES**


## 1) CAC40, DAX40, FTSE MIB40 (Italie), IBEX35 (Espagne), FTSE100 (UK), NASDAQ100 (US), DowJones30 (US), NIKKEI225 (Japon)

In [3]:
# RECUPERATION TIKERS

# Liste des URLs correctement séparées
urls = [
    "https://fr.tradingview.com/symbols/EURONEXT-PX1/components/",  #CAC 40 - France / .PA
    "https://fr.tradingview.com/symbols/XETR-DAX/components/",      #DAX 40 - Allemagne / .DE
    "https://fr.tradingview.com/symbols/INDEX-FTSEMIB/components/", #ftse mib 40 - italie / .MI
    "https://fr.tradingview.com/symbols/BME-IBC/components/",       #IBEX 35 - Espagne / .MC
    "https://fr.tradingview.com/symbols/FTSE-UKX/components/",      #FTSE 100 (que 95/100) - Royaume-Uni / .L
    "https://fr.tradingview.com/symbols/NASDAQ-NDX/components/",    #nasdaq100 - us / rien
    "https://fr.tradingview.com/symbols/DJ-DJI/components/",        #dowjones - us / rien
    #"https://fr.tradingview.com/symbols/TVC-NI225/components/",     #nikkei225 - Japon / .T (Trop de ticker, faire avec Selenium)
]

indice = ["CAC40", "DAX40", "Italie40", "Espagne35", "Angleterre100", "NASDAQ100", "Dow Jones"]

# User-Agent pour éviter d'être bloqué par le site
header = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36"
}

# Dictionnaire pour stocker les tickers de chaque indice
tickers_list_Vfinal = []

for i in urls:
    response = requests.get(i, headers=header)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Sélectionner tous les liens des tickers
    tickers = soup.select("a.tickerNameBox-GrtoTeat")

    # Extraire le texte des tickers
    tickers_list = []
    for i in tickers:
        text_ticker = i.text.strip()
        tickers_list.append(text_ticker)

    tickers_list_Vfinal.append(tickers_list)


# Affichage des tickers avec le nom de l'indice
for index, tickers in enumerate(tickers_list_Vfinal):
    print(f"{indice[index]}: {tickers}")
    print("-"*50)

CAC40: ['MC', 'RMS', 'OR', 'SAN', 'AIR', 'SU', 'EL', 'TTE', 'AI', 'SAF', 'CS', 'BNP', 'DG', 'DSY', 'ACA', 'SGO', 'HO', 'BN', 'ENGI', 'STLAP', 'ORA', 'KER', 'GLE', 'LR', 'RI', 'CAP', 'ML', 'PUB', 'VIE', 'STMPA', 'RNO', 'EN', 'BVI', 'AC', 'URW', 'ERF', 'CA', 'EDEN', 'TEP']
--------------------------------------------------
DAX40: ['SAP', 'SIE', 'DTE', 'ALV', 'AIR', 'MUV2', 'MBG', 'MRK', 'SHL', 'VOW3', 'BMW', 'P911', 'RHM', 'DHL', 'DB1', 'BAS', 'IFX', 'ENR', 'DBK', 'ADS', 'EOAN', 'HNR1', 'DTG', 'HEN3', 'BEI', 'HEI', 'CBK', 'BAYN', 'RWE', 'FRE', 'VNA', 'MTX', 'SRT3', 'CON', 'SY1', 'FME', 'PAH3', 'BNR', 'ZAL', 'QIA']
--------------------------------------------------
Italie40: ['RACE', 'ISP', 'UCG', 'ENEL', 'G', 'ENI', 'STLAM', 'LDO', 'PST', 'STMMI', 'TEN', 'MONC', 'TRN', 'PRY', 'SRG', 'BAMI', 'MB', 'FBK', 'REC', 'BPE', 'UNI', 'BMED', 'BZU', 'INW', 'BMPS', 'CPR', 'BC', 'A2A', 'NEXI', 'TIT', 'PIRC', 'HER', 'DIA', 'IVG', 'IG', 'BPSO', 'AMP', 'SPM', 'IP', 'AZM']
-------------------------------

In [4]:
# CONVERSION EN TIKERS YFINANCE

france_ticker = tickers_list_Vfinal[0]
allemagne_ticker = tickers_list_Vfinal[1]
italie_ticker = tickers_list_Vfinal[2]
espagne_ticker = tickers_list_Vfinal[3]
angleterre_ticker = tickers_list_Vfinal[4]
angleterre_ticker_sans_point = [i.rstrip('.') for i in angleterre_ticker] # Certains tickers finissent par un ".", je le supprime
angleterre_ticker_clean = [ticker.replace('.', '-') for ticker in angleterre_ticker_sans_point] # Un ticker possède un "." en son milieu. je remplace par "-"
nasdaq_ticker = tickers_list_Vfinal[5]
dowjones_ticker = tickers_list_Vfinal[6]
#japon_ticker = tickers_list_Vfinal[7]

france_ticker_yf = [ticker + ".PA" for ticker in france_ticker]
france_ticker_yf.append("MT.AS") # Rajout car seule action pas en .PA
allemagne_ticker_yf= [ticker + ".DE" for ticker in allemagne_ticker]
italie_ticker_yf = [ticker + ".MI" for ticker in italie_ticker]
espagne_ticker_yf = [ticker + ".MC" for ticker in espagne_ticker]
angleterre_ticker_yf = [ticker + ".L" for ticker in angleterre_ticker_clean]
nasdaq_ticker_yf = nasdaq_ticker
dowjones_ticker_yf = dowjones_ticker
#japon_ticker_yf = [ticker + ".T" for ticker in tickers_list_Vfinal[7]]

## 2) BEL20 (Belgique 20), AEX (Pays-Bas 25), OMX Helsinki (Finlande 25), OMX Stockholm 30 (Suède 30), OMX Copenhagen 20 (Danemark 20)

In [9]:
# Scrapper via Yahoo finance

# Liste des URL pour les différents indices
urls = [
    "https://finance.yahoo.com/quote/BEL20.BR/components/",  # BEL20 (Belgique 20) / 19 .BR et .AS
    "https://finance.yahoo.com/quote/%5EAEX/components/",  # AEX (Pays-Bas 25) / 22 .AS
    "https://finance.yahoo.com/quote/%5EOMXH25/components/",  # OMX Helsinki (Finlande 25) / 25 .HE
    "https://finance.yahoo.com/quote/%5EOMX/components/", # OMX Stockholm 30 (Suède 30) / 30 .ST
    "https://finance.yahoo.com/quote/%5EOMXC25/components/" # OMX Copenhagen 20 (Danemark 20) / 20 .CO
]

# En-tête pour éviter le blocage du site
header = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36"}

# Dictionnaire pour stocker les noms des entreprises et les tickers de chaque indice. 
# Clé=nom_indice : valeur=dico dans ce dico, Clé1=nom : valeur1=liste des entreprises et Clé2=tiker : valeur2=liste des tikers
tickers = []

# Boucle sur chaque URL pour récupérer les données du tableau
for url in urls:
    # Envoi de la requête
    reponse = requests.get(url, headers=header) # Capture et lecture de l'url
    reponse.encoding = 'utf-8'
    html = reponse.text # Page html retrancrite en format .text pour récupérer ce qu'on veut
    time.sleep(1) # Delais pour bien que les pages s'affiches
    
    # Récupération du tableau
    df = pd.read_html(html)[0] # Retourne une liste des tables de la page. On veut le 1er
    #display(df)
    
    # Extraction des noms d'entreprises et tickers du tableau et création d'une liste
    ticker = df["Symbol"].tolist()

    # Ajout ticker dans la liste vide tickers
    tickers.append(ticker) # Ajout de la structure du nouveau dico


belgique_ticker = tickers[0]
pays_bas_ticker = tickers[1]
finlande_ticker = tickers[2]
suede_ticker = tickers[3]
danemark_ticker = tickers[4]


/var/folders/t2/7z3y2qws3jx8xljd1wjn53700000gn/T/ipykernel_29559/1539855697.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html)[0] # Retourne une liste des tables de la page. On veut le 1er
/var/folders/t2/7z3y2qws3jx8xljd1wjn53700000gn/T/ipykernel_29559/1539855697.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html)[0] # Retourne une liste des tables de la page. On veut le 1er
/var/folders/t2/7z3y2qws3jx8xljd1wjn53700000gn/T/ipykernel_29559/1539855697.py:28: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html)[0] # Retourne une liste des tables de la p

'\nbelgique_ticker = indices_data["BEL20 : Belgique"]["Ticker"]\npays_bas_ticker = indices_data["AEX25 : Pays-Bas"]["Ticker"]\nfinlande_ticker = indices_data["OMX Helsinki25 : Finlande"]["Ticker"]\nsuede_ticker = indices_data["OMX Stockholm30 : Suède"]["Ticker"]\ndanemark_ticker = indices_data["OMX Copenhagen25 : Danemark"]["Ticker"]\n'

## 3) STOXX50

In [4]:
# Scrappé il y a longtemps
stoxx50_ticker = [
        'MC.PA', 'SAP.DE', 'RMS.PA', 'ASML.AS', 'OR.PA', 'ITX.MC', 'SIE.DE', 
        'DTE.DE', 'SU.PA', 'AIR.PA', 'SAN.PA', 'TTE.PA', 'ALV.DE', 'EL.PA', 
        'SAF.PA', 'AI.PA', 'ABI.BR', 'PRX.AS', 'IBE.MC', 'CS.PA', 'SAN.MC', 
        'ISP.MI', 'RACE.MI', 'BNP.PA', 'MUV2.DE', 'UCG.MI', 'ENEL.MI', 
        'BBVA.MC', 'DG.PA', 'MBG.DE', 'INGA.AS', 'VOW3.DE', 'BMW.DE', 
        'ADYEN.AS', 'ADS.DE', 'SGO.PA', 'DB1.DE', 'BN.PA', 'IFX.DE', 'BAS.DE', 
        'ENI.MI', 'WKL.AS', 'DHL.DE', 'NDA-SE.ST', 'STLAP.PA', 'AD.AS', 'KER.PA', 
        'RI.PA', 'NOKIA.HE', 'BAYN.DE']

## 4) SP500 et NIKKEY225 (Japon 225)

In [2]:
# SP500 VIA PLAYWRIGHT VIA TRAIDINGVIEW

async with async_playwright() as p:
    # Lancer le navigateur Chromium en mode asynchrone car sur Jupytair il faut être asynchrone
    browser = await p.chromium.launch(headless=True)  # headless=False pour voir le navigateur
    page = await browser.new_page() # Ouvre une nouvelle page
    
    await page.goto("https://fr.tradingview.com/symbols/SPX/components/") # Va sur l'URL demandé
    #Boucle pour cliquer sur "Charger plus" 5 fois
    for _ in range(5):
        await page.get_by_role("button", name="Charger plus").click()
        
    # Sélectionner toutes les balises <a> avec la classe spécifique et récupération du text
    sp500_ticker = await page.locator("a.tickerNameBox-GrtoTeat").all_inner_texts()
    print(sp500_ticker)
    
    await browser.close()

['AAPL', 'NVDA', 'MSFT', 'AMZN', 'GOOGL', 'GOOG', 'META', 'BRK.B', 'AVGO', 'TSLA', 'LLY', 'WMT', 'V', 'JPM', 'MA', 'XOM', 'UNH', 'ORCL', 'COST', 'PG', 'JNJ', 'NFLX', 'ABBV', 'HD', 'BAC', 'KO', 'TMUS', 'CVX', 'CRM', 'MRK', 'CSCO', 'PM', 'IBM', 'ABT', 'WFC', 'LIN', 'MCD', 'GE', 'PEP', 'ACN', 'TMO', 'T', 'PLTR', 'VZ', 'AXP', 'MS', 'DIS', 'RTX', 'ISRG', 'QCOM', 'AMGN', 'NOW', 'BX', 'ADBE', 'PGR', 'GS', 'INTU', 'CAT', 'AMD', 'TXN', 'NEE', 'SPGI', 'DHR', 'UBER', 'PFE', 'UNP', 'GILD', 'BKNG', 'BSX', 'BLK', 'SYK', 'SCHW', 'HON', 'CMCSA', 'VRTX', 'TJX', 'DE', 'C', 'LOW', 'COP', 'BMY', 'AMAT', 'BA', 'MDT', 'ADP', 'FI', 'PANW', 'CB', 'MMC', 'ETN', 'LMT', 'SBUX', 'NKE', 'MU', 'PLD', 'INTC', 'ADI', 'ANET', 'MO', 'UPS', 'SO', 'ICE', 'AMT', 'LRCX', 'KKR', 'ELV', 'CME', 'WELL', 'DUK', 'KLAC', 'WM', 'SHW', 'CI', 'AON', 'MDLZ', 'CVS', 'GEV', 'CRWD', 'AJG', 'EQIX', 'MCK', 'MMM', 'MCO', 'HCA', 'CTAS', 'PH', 'TT', 'ABNB', 'APH', 'TDG', 'REGN', 'APO', 'ORLY', 'ITW', 'CL', 'FTNT', 'ZTS', 'RSG', 'ECL', 'NOC',

In [49]:
# NIKKEY225 VIA PLAYWRIGHT VIA TRAIDINGVIEW

async with async_playwright() as p:
    # Lancer le navigateur Chromium en mode asynchrone car sur Jupytair il faut être asynchrone
    browser = await p.chromium.launch(headless=True)  # headless=False pour voir le navigateur
    page = await browser.new_page() # Ouvre une nouvelle page
    
    await page.goto("https://fr.tradingview.com/symbols/TVC-NI225/components/") # Va sur l'URL demandé
    #Boucle pour cliquer sur "Charger plus" 2 fois
    for _ in range(2):
        await page.get_by_role("button", name="Charger plus").click()
        
    # Sélectionner toutes les balises <a> avec la classe spécifique et récupération du text
    japon_ticker = await page.locator("a.tickerNameBox-GrtoTeat").all_inner_texts()
    japon_ticker_yf = [ticker + ".T" for ticker in japon_ticker]
    
    await browser.close()

['7203.T', '6758.T', '8306.T', '6501.T', '8316.T', '6861.T', '9983.T', '6098.T', '7974.T', '9432.T', '4519.T', '9984.T', '8411.T', '8766.T', '8035.T', '9433.T', '9434.T', '8058.T', '8001.T', '4063.T', '8031.T', '4502.T', '7267.T', '7011.T', '2914.T', '4568.T', '7741.T', '6857.T', '3382.T', '6702.T', '6902.T', '6178.T', '4661.T', '8725.T', '7751.T', '6503.T', '6981.T', '6367.T', '6723.T', '6752.T', '8630.T', '8750.T', '6301.T', '5108.T', '6146.T', '8053.T', '6954.T', '4543.T', '8002.T', '4578.T', '9613.T', '6701.T', '4901.T', '8801.T', '6273.T', '7269.T', '4689.T', '5401.T', '9020.T', '8591.T', '7832.T', '1925.T', '6594.T', '8267.T', '6762.T', '2802.T', '4452.T', '9022.T', '8604.T', '2502.T', '4307.T', '8309.T', '8802.T', '8308.T', '8015.T', '4503.T', '9766.T', '8830.T', '9101.T', '5020.T', '6971.T', '7733.T', '9735.T', '1605.T', '1928.T', '6326.T', '6988.T', '5802.T', '7270.T', '9104.T', '4755.T', '4507.T', '9503.T', '9531.T', '9843.T', '5803.T', '3659.T', '2503.T', '8697.T', '8795.T',

# **II- MISE EN PLACE DE SCRIPT GENERAL POUR CREATIONS DE DATA FRAME**

In [ ]:
def get_stock_data(tickers, indice_name, Ticker_Yahoo_Indice, file_path):
    
    # Création du DataFrame initial avec une colonne vide
    df = pd.DataFrame({'Ticker_Yahoo_Finance': tickers})

    # Récupérer toutes les données en une seule requête
    tickers_obj = yf.Tickers(" ".join(tickers))  # Récupération groupée des tickers

    # Initialisation des listes pour stocker les données
    noms_entreprises = []
    pays = []
    place_boursiere = []
    secteur_activite = []
    capitalisation_boursiere = []
    tickers_sans_suffixe = []

    for i in tickers:
        try:
            # Récupérer les informations via l'objet Tickers
            info = tickers_obj.tickers[i].info

            # Extraction du ticker sans suffixe
            symbole_sans_suffixe = i.split(".")[0]
            tickers_sans_suffixe.append(symbole_sans_suffixe)

            # Ajouter les informations dans leurs listes respectives
            noms_entreprises.append(info.get('shortName', 'Non disponible'))
            pays.append(info.get('country', 'Non disponible'))
            place_boursiere.append(info.get('exchange', 'Non disponible'))
            secteur_activite.append(info.get('sector', 'Non disponible'))
            capitalisation_boursiere.append(info.get('marketCap', None))

        except Exception as e:
            print(f"Erreur lors de la récupération des données pour {i}: {e}")
            noms_entreprises.append('Non disponible')
            pays.append('Non disponible')
            place_boursiere.append('Non disponible')
            secteur_activite.append('Non disponible')
            capitalisation_boursiere.append(None)
            tickers_sans_suffixe.append(i)  # Ajouter le ticker original en cas d'erreur

        time.sleep(0.5)  # Petite pause pour éviter le blocage

    # Ajout des colonnes au DataFrame
    df['Nom_Entreprise'] = noms_entreprises
    df['Pays'] = pays
    df['Place_Boursiere'] = place_boursiere
    df['Secteur_Activite'] = secteur_activite
    df['Capitalisation_Boursiere'] = capitalisation_boursiere
    df['Ticker'] = tickers_sans_suffixe
    df['Nom_Indice'] = indice_name
    df['Nombres_Entreprises'] = len(noms_entreprises)
    df['Ticker_Indice_Yahoo'] = Ticker_Yahoo_Indice

    # Calcul de la capitalisation boursière totale
    total_capitalisation = df['Capitalisation_Boursiere'].fillna(0).sum()

    # Calcul pondération
    df['Ponderation'] = round((df['Capitalisation_Boursiere'] / total_capitalisation) * 100, 2)

    # Réorganisation des colonnes
    df = df[
        ['Nom_Entreprise', 'Ticker', 'Ticker_Yahoo_Finance', 'Ponderation', 'Secteur_Activite', 'Nom_Indice',
         'Ticker_Indice_Yahoo', 'Pays', 'Place_Boursiere', 'Nombres_Entreprises', 'Capitalisation_Boursiere']
    ]

    # Exportation en CSV
    df.to_csv(file_path, index=False, encoding='utf-8')

    return df


# **III- RECUPERATION DES DONNEES EN CSV**

In [ ]:
df_cac40 = get_stock_data(france_ticker_yf, 'CAC 40', '^FCHI', 'composition_france.csv')
df_dax40 = get_stock_data(allemagne_ticker_yf, 'DAX', '^GDAXI', 'composition_dax.csv')
df_italie40 = get_stock_data(italie_ticker_yf, 'FTSE MIB Index', 'FTSEMIB.MI', 'composition_italie.csv')
df_espagne35 = get_stock_data(espagne_ticker_yf, 'IBEX 35', '^IBEX', 'composition_espagne.csv')
df_angleterre = get_stock_data(angleterre_ticker_yf, 'FTSE 100', '^FTSE', 'composition_angleterre.csv')
df_nasdaq100 = get_stock_data(nasdaq_ticker_yf, 'NASDAQ 100', '^NDX',  'composition_nasdaq100.csv')
df_dowjones = get_stock_data(dowjones_ticker_yf, 'Dow Jones 30', '^DJI', 'composition_dowjones.csv')

df_belgique = get_stock_data(belgique_ticker, 'BEL 20', '^BFX', 'composition_belgiqu0.csv')
df_pays_bas = get_stock_data(pays_bas_ticker, 'AEX-Index', '^AEX', 'composition_paysbas.csv')
df_finlande = get_stock_data(finlande_ticker, 'OMX Helsinki 25', '^OMXH25', 'composition_finlande.csv')
df_suede = get_stock_data(suede_ticker, 'OMX Stockholm 30', '^OMXS30', 'composition_suede.csv')
df_danemark = get_stock_data(danemark_ticker, 'OMX Copenhagen 25', '^OMXC25', 'composition_danemark.csv')

df_stoxx50 = get_stock_data(stoxx50_ticker, 'STOXX 50', '^STOXX50E', 'composition_europe50.csv')

df_sp500 = get_stock_data(sp500_ticker, 'S&P 500', '^GSPC', 'composition_sp500.csv')
df_japon225 = get_stock_data(japon_ticker_yf, 'Nikkei 225', '^N225', 'composition_japon.csv')